# Daily Sales Time-Series Analysis

## Project Overview
Analyze daily sales data for trend, seasonality, holiday effects, moving averages, and structural breaks before forecasting.

## Learning Objectives
- Decompose a sales time series into trend, seasonality, and residual
- Detect and visualize structural breaks and regime changes
- Analyze holiday and day-of-week effects
- Apply moving averages and rolling statistics for smoothing
- Assess stationarity using the ADF test

## Problem Statement
Retail planners need to understand the components of daily sales — trend direction, weekly/annual seasonality, and holiday spikes — to make informed inventory and staffing decisions.

## Why This Project Matters
Daily sales analysis is the foundation of demand planning. Identifying patterns before building forecasting models ensures that models capture the right signals and that business users trust the results.

## Dataset Overview
Synthetic daily sales dataset: 3 years (~1,095 days) with trend, weekly seasonality, annual seasonality, holiday spikes, and noise.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_days = 1095
dates = pd.date_range('2021-01-01', periods=n_days, freq='D')
t = np.arange(n_days)

# Components
trend = 5000 + t * 2.5  # upward trend
weekly = 400 * np.sin(2 * np.pi * t / 7)  # weekly cycle
annual = 800 * np.sin(2 * np.pi * (t - 80) / 365.25)  # annual cycle peaking ~spring
noise = np.random.normal(0, 300, n_days)

# Holiday spikes (Christmas, Black Friday, New Year)
holiday_effect = np.zeros(n_days)
for yr in [2021, 2022, 2023]:
    for m, d, boost in [(12, 25, 3000), (11, 26, 4000), (1, 1, 1500), (7, 4, 1200)]:
        try:
            idx = (pd.Timestamp(yr, m, d) - dates[0]).days
            if 0 <= idx < n_days:
                holiday_effect[idx] = boost
                if idx + 1 < n_days:
                    holiday_effect[idx + 1] = boost * 0.4
        except Exception:
            pass

# Structural break at day 600 (price increase → sales dip)
structural = np.where(t >= 600, -800, 0)

sales = np.clip(trend + weekly + annual + holiday_effect + structural + noise, 500, None).round(0)

df = pd.DataFrame({'Date': dates, 'Sales': sales})
df = df.set_index('Date')
df['DayOfWeek'] = df.index.day_name()
df['Month'] = df.index.month
df['Year'] = df.index.year
df['DayOfYear'] = df.index.dayofyear

print(f'Dataset: {df.shape}')
print(f'Date range: {df.index.min().date()} to {df.index.max().date()}')
print(f'Sales range: {df["Sales"].min():,.0f} — {df["Sales"].max():,.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Missing dates: {pd.date_range(df.index.min(), df.index.max()).difference(df.index).size}')
print(f'\nSales stats:\n{df["Sales"].describe().round(0)}')

## Overall Sales Trend

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, df['Sales'], alpha=0.4, linewidth=0.5, label='Daily')
ax.plot(df.index, df['Sales'].rolling(30).mean(), color='red', linewidth=2, label='30-day MA')
ax.plot(df.index, df['Sales'].rolling(90).mean(), color='darkblue', linewidth=2, label='90-day MA')
ax.axvline(pd.Timestamp('2022-08-19'), color='green', linestyle='--', alpha=0.7, label='Structural break (~day 600)')
ax.set_title('Daily Sales with Moving Averages')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'sales_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Seasonal Decomposition

In [ ]:
decomp = seasonal_decompose(df['Sales'], model='additive', period=7)
fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed')
decomp.trend.plot(ax=axes[1], title='Trend')
decomp.seasonal.plot(ax=axes[2], title='Weekly Seasonality')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'decomposition.png'), dpi=100, bbox_inches='tight')
plt.show()

## Day-of-Week Effect

In [ ]:
dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fig, ax = plt.subplots(figsize=(10, 5))
df.groupby('DayOfWeek')['Sales'].mean().reindex(dow_order).plot.bar(ax=ax, edgecolor='black', color='steelblue')
ax.set_title('Average Sales by Day of Week')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'day_of_week.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Seasonality

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
monthly = df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()
for yr in monthly['Year'].unique():
    sub = monthly[monthly['Year'] == yr]
    ax.plot(sub['Month'], sub['Sales'], marker='o', label=str(yr))
ax.set_title('Monthly Total Sales by Year')
ax.set_xlabel('Month')
ax.set_ylabel('Total Sales')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_seasonality.png'), dpi=100, bbox_inches='tight')
plt.show()

## Stationarity Test (ADF)

In [ ]:
result = adfuller(df['Sales'].dropna())
print(f'ADF Statistic: {result[0]:.4f}')
print(f'p-value: {result[1]:.6f}')
print(f'Critical Values: {result[4]}')
if result[1] < 0.05:
    print('=> Series is stationary (reject null)')
else:
    print('=> Series is non-stationary (fail to reject null)')

## Rolling Volatility

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
rolling_std = df['Sales'].rolling(30).std()
rolling_std.plot(ax=ax, color='coral')
ax.set_title('30-Day Rolling Standard Deviation (Volatility)')
ax.set_ylabel('Std Dev')
ax.axvline(pd.Timestamp('2022-08-19'), color='green', linestyle='--', alpha=0.7, label='Structural break')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'rolling_volatility.png'), dpi=100, bbox_inches='tight')
plt.show()

## Holiday Effect Analysis

In [ ]:
# Flag holiday windows
holidays = []
for yr in [2021, 2022, 2023]:
    for m, d, name in [(12, 25, 'Christmas'), (11, 26, 'BlackFriday'), (1, 1, 'NewYear'), (7, 4, 'July4th')]:
        try:
            holidays.append({'Date': pd.Timestamp(yr, m, d), 'Holiday': name})
        except Exception:
            pass
df_holidays = pd.DataFrame(holidays).set_index('Date')
df_with_h = df.join(df_holidays, how='left')
df_with_h['IsHoliday'] = df_with_h['Holiday'].notna()

print(f'Holiday days: {df_with_h["IsHoliday"].sum()}')
print(f'Avg sales (holiday): {df_with_h.loc[df_with_h["IsHoliday"], "Sales"].mean():,.0f}')
print(f'Avg sales (non-holiday): {df_with_h.loc[~df_with_h["IsHoliday"], "Sales"].mean():,.0f}')
print(f'Holiday lift: {df_with_h.loc[df_with_h["IsHoliday"], "Sales"].mean() - df_with_h.loc[~df_with_h["IsHoliday"], "Sales"].mean():+,.0f}')

## Interpretation and Business Insight
- **Upward trend** is clear — sales grow ~$2.50/day on average
- **Weekly seasonality** shows consistent day-of-week patterns useful for staffing
- **Annual seasonality** peaks in spring/summer and dips in winter (outside holidays)
- **Holiday spikes** are dramatic — Black Friday and Christmas drive 40-60% above average
- **Structural break** around mid-2022 appears as a level shift — likely a price change or market event
- **Rolling volatility** increases around holidays and the structural break
- The series is non-stationary due to trend — differencing or detrending is needed before ARIMA-style forecasting

## Limitations
- Synthetic data with known components
- No product-level granularity
- No external regressors (marketing spend, weather)
- No promotional calendar detail
- Single channel — real retail is multi-channel

## How to Improve This Project
- Add product category and channel breakdowns
- Include marketing spend and promotional calendar as regressors
- Build forecasting models (ARIMA, Prophet, LightGBM with lag features)
- Add anomaly detection for unexpected dips/spikes
- Include weather data for weather-sensitive categories

## Production Considerations
- Daily demand forecasting pipelines feeding inventory systems
- Automated anomaly alerts for sales deviations
- Holiday calendar integration for promotional planning
- Multi-step forecast horizons (7-day, 30-day, 90-day)

## Common Mistakes
- Fitting models without decomposing to understand components first
- Ignoring structural breaks that invalidate historical patterns
- Using annual seasonality period (365) for decomposition on short series
- Not accounting for holiday effects in train/test splits

## Mini Challenge / Exercises
1. Calculate the year-over-year growth rate for total annual sales.
2. What day of the week has the highest sales variance?
3. Apply a 7-day differencing to remove weekly seasonality — is the differenced series stationary?
4. Estimate the magnitude of the structural break by comparing mean sales before and after day 600.

## Final Summary / Key Takeaways
- Daily sales time series contains trend, weekly seasonality, annual seasonality, holiday effects, and a structural break
- Decomposition is essential before any forecasting work
- Moving averages reveal the underlying trend and smooth noise
- Stationarity testing guides the choice of forecasting model
- Holiday and event effects must be modeled explicitly for accurate forecasts